In [1]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [2]:
level = "../../../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_moving_gaussian.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [3]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [4]:
function evolution_at_param(param, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current ghostly coupling: ", param, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = 1.;
    mchi2 = 1.;
    c4 = param;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = 4; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 2^2;
    k0chi = k0phi;
    x0phi = 0.3;
    x0chi = 0.7;

    offsetphi = 0;
    offsetchi = 0;

    aStochastic = 0;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (e-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamPhiTab[end][1:Int(div(length(hamPhiTab[end]),4))+1])), 
        hamPhiTab[end]
    )
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamChiTab[end][1:Int(div(length(hamChiTab[end]),4))+1])), 
        hamChiTab[end]
    )
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end 
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(param,".jld2")) param stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_param (generic function with 1 method)

### main()

In [5]:
#param_base = 1.2
#param_table = reverse([param_base^i for i in -16:12])

param_table = [invC4 for invC4 in 1:8:128]
param_table = param_table.^(-1)

16-element Vector{Float64}:
 1.0
 0.1111111111111111
 0.058823529411764705
 0.04
 0.030303030303030304
 0.024390243902439025
 0.02040816326530612
 0.017543859649122806
 0.015384615384615385
 0.0136986301369863
 0.012345679012345678
 0.011235955056179775
 0.010309278350515464
 0.009523809523809525
 0.008849557522123894
 0.008264462809917356

In [6]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 11;
    max_target_time = 2 * 10^4;
    # ... and their initial values
    current_res_log2 = 10;
    current_target_time = 10;
    
    # initialise the runaway time for handover to next param value
    runaway_time = Inf;
    
    # set table of desired param_table (NOTE: links to scaling assumption below)
    param_base = 1
    param_table = [invC4 for invC4 in 1:4:128]
    param_table = param_table.^(-1)
    
    # loop over all values in param_table
    for param in param_table
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_param(
                param, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("PARAM = ", param, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next param value from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(param_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [ ]:
main()

persistent random seed: 3605740
current ghostly coupling: 1.0
current resolution: 256
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
  5.781214 seconds (5.00 M allocations: 2.853 GiB, 4.86% gc time, 73.34% compilation time)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
  4.983716 seconds (3.73 M allocations: 9.955 GiB, 7.21% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
 19.074588 seconds (7.42 M allocations: 38.654 GiB, 11.68% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=4.21
Runaway detected at time t=6.05
Finished plotting.


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/01_trivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/04_coupling/01_Gauss/plots/3605740/1.0/animation_Nx=1024.gif


Saved data.
Increasing resolution from N = 10 to 11
persistent random seed: 3605740
current ghostly coupling: 1.0
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
  5.102176 seconds (3.73 M allocations: 9.955 GiB, 8.92% gc time, 0.21% compilation time: 100% of which was recompilation)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
 18.529513 seconds (7.42 M allocations: 38.654 GiB, 6.70% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
 66.794310 seconds (21.35 M allocations: 151.815 GiB, 8.12% gc time)
... terminated
Output plot directory already exists.
Convergence lost at time t=9.

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/01_trivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/04_coupling/01_Gauss/plots/3605740/1.0/animation_Nx=2048.gif


persistent random seed: 3605740
current ghostly coupling: 0.2
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
 56.700316 seconds (6.18 M allocations: 16.317 GiB, 2.71% gc time, 0.21% compilation time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
 61.812864 seconds (12.17 M allocations: 63.455 GiB, 3.06% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
123.104693 seconds (35.08 M allocations: 249.442 GiB, 6.12% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
No runaway detected.


[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/01_trivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/04_coupling/01_Gauss/plots/3605740/0.2/animation_Nx=2048.gif


Finished plotting.
Saved data.
Increasing target time to T = 65.78242024870889
persistent random seed: 3605740
current ghostly coupling: 0.2
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
Terminating because one of the fields grew too large at time t = 36.24804687502036.
 27.418451 seconds (13.39 M allocations: 35.804 GiB, 12.11% gc time, 0.86% compilation time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
Terminating because one of the fields grew too large at time t = 40.00048828094184.
 82.597566 seconds (29.52 M allocations: 154.017 GiB, 4.02% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/01_trivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/04_coupling/01_Gauss/plots/3605740/0.2/animation_Nx=2048.gif


Saved data.
Target time reset to confidently detected runaway time T = 21.247721740332974
PARAM = 0.2 DONE!
Updating target time for next param value from T = 21.247721740332974 ... to T = 57.75729590290132
persistent random seed: 3605740
current ghostly coupling: 0.1111111111111111
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9992290114686284
		max |amplitude| chi before rescaling: 3.9992290114686284
Terminating because one of the fields grew too large at time t = 43.13203124992018.
 33.054584 seconds (15.93 M allocations: 42.611 GiB, 4.94% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.9999518088625607
		max |amplitude| chi before rescaling: 3.9999518088625607
Terminating because one of the fields grew too large at time t = 42.44072265590633.
 90.361999 seconds (31.32 M allocations: 163.428 GiB, 3.96% gc time)
... terminated
current resolution: 2048
	user-assigne

### export .jl for production run

In [ ]:
using NBInclude
nbexport("main.jl", "main.ipynb")